# EV 에너지 모델 학습

모델을 학습하고 `models/energy_model.joblib`로 저장합니다. 서비스 실행 시 이 파일을 재사용합니다.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA_PATH = ROOT / 'datas' / 'ev_energy_consumption.csv'
MODEL_PATH = ROOT / 'models' / 'energy_model.joblib'
TARGET = 'energy_consumption_kwhper100km'
BASE_FEATURES = ['speed_kmh', 'payload_kg', 'ambient_temp_C', 'hvac_power_kw', 'road_grade_pct', 'battery_temp_C', 'driving_style_index', 'tire_pressure_bar', 'trip_distance_km']

def create_ev_features(data):
    result = data.copy()
    result['speed_squared'] = result['speed_kmh'] ** 2
    result['ambient_temp_deviation'] = (result['ambient_temp_C'] - 25).abs()
    result['battery_temp_deviation'] = (result['battery_temp_C'] - 30).abs()
    result['tire_pressure_deviation'] = (result['tire_pressure_bar'] - 2.4).abs()
    result['payload_grade'] = result['payload_kg'] * result['road_grade_pct']
    result['hvac_temp_interaction'] = result['hvac_power_kw'] * result['ambient_temp_deviation']
    result['speed_driving_interaction'] = result['speed_kmh'] * result['driving_style_index']
    return result

data = pd.read_csv(DATA_PATH).dropna(subset=BASE_FEATURES + [TARGET])
engineered = create_ev_features(data)
features = [c for c in engineered.columns if c != TARGET]
x_train, x_test, y_train, y_test = train_test_split(engineered[features], data[TARGET], test_size=0.2, random_state=42)
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(x_train, y_train)
predictions = model.predict(x_test)
metrics = {'R²': r2_score(y_test, predictions), 'MAE': mean_absolute_error(y_test, predictions), 'RMSE': np.sqrt(mean_squared_error(y_test, predictions))}
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': model, 'metrics': metrics}, MODEL_PATH)
print(f'저장 완료: {MODEL_PATH}')
print(metrics)

저장 완료: /home/aiuser/E_car/models/energy_model.joblib
{'R²': 0.9417201218050536, 'MAE': 0.7045131976291655, 'RMSE': np.float64(0.8874742526957434)}
